In [21]:
import psycopg2

In [22]:
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [23]:
cur.execute('SELECT "30pid_superfamily_id", centroid_orf_id FROM "30pid_superfamily_clusters"')

In [ ]:
rows = cur.fetchall()
print(f"Fetched {len(rows)} rows")

Fetched 22235 rows
[(1, 17), (2, 35), (3, 37)]


In [25]:
def parse_fasta(path):
    seqs = {}
    with open(path) as f:
        header, seq_parts = None, []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header is not None:
                    orf_id = int(header.split('|')[0])
                    seqs[orf_id] = ''.join(seq_parts)
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line)
        if header is not None:
            orf_id = int(header.split('|')[0])
            seqs[orf_id] = ''.join(seq_parts)
    return seqs

fasta_seqs = parse_fasta('petadex_blastnr_orfs.fa')
print(f"Parsed {len(fasta_seqs)} sequences")

Parsed 5593336 sequences


In [26]:
import pandas as pd

df = pd.DataFrame(rows, columns=['superfamily_id', 'centroid_orf_id'])
df['sequence'] = df['centroid_orf_id'].map(fasta_seqs)

print(df.head())
print(f"\nMissing sequences: {df['sequence'].isna().sum()} / {len(df)}")

   superfamily_id  centroid_orf_id  \
0               1               17   
1               2               35   
2               3               37   
3               4               40   
4               5               65   

                                            sequence  
0  MELNSMQFLLGLIGLLLLIVTSLRRWLLRRESPQKQAVDFHGELYQ...  
1  MPTEVVMPKWGLSMQEGKINLWLKREGEAVQKGEPIAEVETEKITN...  
2  MIRPVTFRNMNQQIIGILHTPDNIRLNEKVPGILMFHGFTGNKTEA...  
3  MSVSLKGALRVLPLAASVALVGCFGGGDPEPGPDGQALTNPGEYEI...  
4  MDENYPVLPGADSFFIKGNEIGILISHGFNGTPQSVRFLGRAMASD...  

Missing sequences: 17959 / 22235


In [27]:
df

,superfamily_id,centroid_orf_id,sequence
0,1,17,MELNSMQFLLGLIGLLLLIVTSLRRWLLRRESPQKQAVDFHGELYQ...
1,2,35,MPTEVVMPKWGLSMQEGKINLWLKREGEAVQKGEPIAEVETEKITN...
2,3,37,MIRPVTFRNMNQQIIGILHTPDNIRLNEKVPGILMFHGFTGNKTEA...
3,4,40,MSVSLKGALRVLPLAASVALVGCFGGGDPEPGPDGQALTNPGEYEI...
4,5,65,MDENYPVLPGADSFFIKGNEIGILISHGFNGTPQSVRFLGRAMASD...
...,...,...,...
22230,22231,1373065406,NaN
22231,22232,1373067553,NaN
22232,22233,1373068148,NaN
22233,22234,1373086334,NaN
